# Comprehensive Model Comparison Analysis
## Dengue Prediction - Ward and Monthly Aggregation

**Objective:** Compare predictions from all trained models (Linear Regression, GBM, Random Forest, XGBoost, MLP) at ward and monthly levels to identify model agreement patterns and consistency.


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)
print("All libraries loaded successfully!")


All libraries loaded successfully!


## Section 1: Data Preparation and Feature Engineering

In [ ]:
# Load Dataset
df = pd.read_csv('bengaluru_wards_dataset.csv')

if 'Date' not in df.columns:
    df['Date'] = pd.to_datetime(df[['Year','Month']].assign(DAY=1))
else:
    df['Date'] = pd.to_datetime(df['Date'])

print(f'Dataset loaded: {len(df)} rows, columns: {list(df.columns)[:10]}...')

# Feature Engineering
df_fe = df.sort_values(['Ward_ID','Date']).reset_index(drop=True).copy()

# Lag features
df_fe['Rainfall_Lag1'] = df_fe.groupby('Ward_ID')['Rainfall_mm'].shift(1)
df_fe['Rainfall_Lag2'] = df_fe.groupby('Ward_ID')['Rainfall_mm'].shift(2)
df_fe['Temp_Lag1'] = df_fe.groupby('Ward_ID')['Avg_Temp_C'].shift(1)
df_fe['Cases_Lag1'] = df_fe.groupby('Ward_ID')['Dengue_Cases'].shift(1)

# Rolling means
df_fe['Rainfall_roll3_mean'] = df_fe.groupby('Ward_ID')['Rainfall_mm'].rolling(window=3, min_periods=1).mean().reset_index(0,drop=True)
df_fe['Cases_roll3_mean'] = df_fe.groupby('Ward_ID')['Dengue_Cases'].rolling(window=3, min_periods=1).mean().reset_index(0,drop=True)

# Temporal features
df_fe['Month'] = df_fe['Date'].dt.month
df_fe['Year'] = df_fe['Date'].dt.year
df_fe['Is_Monsoon'] = df_fe['Month'].isin([6,7,8,9]).astype(int)

# Ward-level aggregates
ward_agg = df_fe.groupby('Ward_ID')[['Garbage_Complaints','Waterlogging_Complaints']].mean().rename(columns=lambda x: x+'_ward_mean')
df_fe = df_fe.merge(ward_agg, left_on='Ward_ID', right_index=True)
df_fe = df_fe.dropna().reset_index(drop=True)

print(f'After feature engineering: {len(df_fe)} rows')

# Prepare features and target
features = [
    'Rainfall_mm','Avg_Temp_C','Garbage_Complaints','Waterlogging_Complaints',
    'Rainfall_Lag1','Rainfall_Lag2','Temp_Lag1','Cases_Lag1',
    'Rainfall_roll3_mean','Cases_roll3_mean','Is_Monsoon','Garbage_Complaints_ward_mean','Waterlogging_Complaints_ward_mean'
]
X = df_fe[features].copy()
y = df_fe['Dengue_Cases'].copy()

# Train/Test Split (80/20 chronological)
split_idx = int(len(X) * 0.8)
X_test = X.iloc[split_idx:].copy()
y_test = y.iloc[split_idx:].copy()

print(f'Test set size: {len(X_test)} samples')
print(f'Features prepared: {len(features)} features')


Dataset loaded: 7128 rows, columns: ['Ward_ID', 'Year', 'Month', 'Rainfall_mm', 'Avg_Temp_C', 'Garbage_Complaints', 'Waterlogging_Complaints', 'Dengue_Cases', 'Risk_Level', 'Date']...
After feature engineering: 6732 rows
Test set size: 1347 samples
Features prepared: 13 features


## Section 2: Load All Trained Models and Generate Predictions

In [11]:
print('\n' + '='*100)
print('LOADING ALL TRAINED MODELS')
print('='*100)

models_info = {
    'Linear Regression': 'model_linear_regression.joblib',
    'GBM': 'model_gbm.joblib',
    'Random Forest': 'model_random_forest.joblib',
    'XGBoost': 'model_xgboost.joblib',
    'MLP': 'model_mlp.joblib'
}

scalers_info = {
    'Linear Regression': 'scaler_linear_regression.joblib',
    'MLP': 'scaler_mlp.joblib'
}

all_model_predictions = {}
successfully_loaded = []
all_scalers = {}

# Load scalers first
for model_name, scaler_file in scalers_info.items():
    try:
        scaler = joblib.load(scaler_file)
        all_scalers[model_name] = scaler
        print(f'✓ {model_name} scaler loaded')
    except Exception as e:
        print(f'✗ {model_name} scaler not found: {str(e)[:50]}')

# Load models and generate predictions
for model_name, model_file in models_info.items():
    try:
        model = joblib.load(model_file)
        
        # Apply scaling if available for this model
        if model_name in all_scalers:
            X_test_scaled = pd.DataFrame(all_scalers[model_name].transform(X_test), 
                                        columns=X_test.columns, index=X_test.index)
            predictions = model.predict(X_test_scaled)
            print(f'✓ {model_name:20s} - Loaded successfully (with scaling), predictions generated')
        else:
            predictions = model.predict(X_test)
            print(f'✓ {model_name:20s} - Loaded successfully (no scaling), predictions generated')
        
        all_model_predictions[model_name] = predictions
        successfully_loaded.append(model_name)
    except Exception as e:
        print(f'✗ {model_name:20s} - Error: {str(e)[:50]}')

print(f'\n✓ Successfully loaded {len(successfully_loaded)}/{len(models_info)} models')
print('='*100)


LOADING ALL TRAINED MODELS
✓ Linear Regression scaler loaded
✓ MLP scaler loaded
✓ Linear Regression    - Loaded successfully (with scaling), predictions generated
✓ GBM                  - Loaded successfully (no scaling), predictions generated
✓ Random Forest        - Loaded successfully (no scaling), predictions generated
✓ XGBoost              - Loaded successfully (no scaling), predictions generated
✓ MLP                  - Loaded successfully (with scaling), predictions generated

✓ Successfully loaded 5/5 models


## Section 3: Create Ward-Month Aggregation Table with All Model Predictions

**Description:** Comprehensive table showing actual and predicted dengue cases by ward and month for all models

In [13]:
print('\n' + '='*150)
print('BUILDING COMPREHENSIVE COMPARISON TABLE: WARD x MONTH x ALL MODELS')
print('='*150)

# Prepare test data with date information
test_data_full = df_fe.iloc[split_idx:].copy().reset_index(drop=True)
test_data_full['Actual_Cases'] = y_test.values

# Add predictions from all models
for model_name, predictions in all_model_predictions.items():
    test_data_full[f'{model_name}_Pred'] = predictions

# Aggregate by Ward, Year, Month
pred_columns = [f'{model}_Pred' for model in successfully_loaded]
agg_dict = {'Actual_Cases': 'mean'}
agg_dict.update({col: 'mean' for col in pred_columns})

comparison_table = test_data_full.groupby(['Ward_ID', 'Year', 'Month']).agg(agg_dict).reset_index()

# Rename columns for clarity
comparison_table = comparison_table.rename(columns=lambda x: x.replace('_Pred', '') if '_Pred' in x else x)

# Sort by Ward, Year, Month
comparison_table = comparison_table.sort_values(['Ward_ID', 'Year', 'Month']).reset_index(drop=True)

# Round for display
for col in comparison_table.columns:
    if col not in ['Ward_ID', 'Year', 'Month']:
        comparison_table[col] = comparison_table[col].round(2)

print(f'\n✓ Comparison table created:')
print(f'  - Rows: {comparison_table.shape[0]} (unique ward-month-year combinations)')
print(f'  - Columns: {comparison_table.shape[1]}')
print(f'  - Wards: {comparison_table["Ward_ID"].nunique()}')
print(f'  - Years: {comparison_table["Year"].nunique()}')
print(f'  - Months: {comparison_table["Month"].nunique()}')
print(f'\nColumns: {", ".join(comparison_table.columns.tolist())}')
print('\n' + '='*150)
print('SAMPLE OF COMPARISON TABLE (First 15 rows):')
print('='*150)
print(comparison_table.head(15).to_string(index=False))
print('='*150)

# Save table to CSV
comparison_table.to_csv('model_comparison_ward_month_table.csv', index=False)
print('\n✓ Comparison table saved to: model_comparison_ward_month_table.csv')



BUILDING COMPREHENSIVE COMPARISON TABLE: WARD x MONTH x ALL MODELS

✓ Comparison table created:
  - Rows: 1347 (unique ward-month-year combinations)
  - Columns: 9
  - Wards: 40
  - Years: 3
  - Months: 12

Columns: Ward_ID, Year, Month, Actual_Cases, Linear Regression, GBM, Random Forest, XGBoost, MLP

SAMPLE OF COMPARISON TABLE (First 15 rows):
 Ward_ID  Year  Month  Actual_Cases  Linear Regression   GBM  Random Forest   XGBoost   MLP
     159  2022      4          25.0              18.31 18.44          15.94 18.309999 19.04
     159  2022      5          19.0              22.63 22.40          21.78 21.200001 19.69
     159  2022      6          38.0              38.57 38.80          40.41 37.400002 40.57
     159  2022      7          57.0              57.67 58.64          59.75 58.270000 57.81
     159  2022      8          65.0              61.47 61.97          60.06 61.720001 60.54
     159  2022      9          74.0              78.96 77.88          77.45 78.660004 77.75
     1

## Section 4: Model Performance Comparison Across All Metrics

In [12]:
print('\n' + '='*100)
print('MODEL PERFORMANCE METRICS COMPARISON')
print('='*100)

# Calculate metrics for each model
model_metrics = []
for model_name in successfully_loaded:
    predictions = all_model_predictions[model_name]
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    model_metrics.append({
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    })

model_metrics_df = pd.DataFrame(model_metrics).sort_values('R²', ascending=False)

print('\nOVERALL MODEL PERFORMANCE METRICS:')
print(model_metrics_df.to_string(index=False))
print('='*100)

# Create visualization with reduced size
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = plt.cm.Set3(np.linspace(0, 1, len(successfully_loaded)))

# Plot 1: RMSE
axes[0, 0].bar(model_metrics_df['Model'], model_metrics_df['RMSE'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 0].set_title('Root Mean Squared Error (RMSE) - Lower is Better', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('RMSE', fontsize=10)
axes[0, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(model_metrics_df['RMSE']):
    axes[0, 0].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Plot 2: MAE
axes[0, 1].bar(model_metrics_df['Model'], model_metrics_df['MAE'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 1].set_title('Mean Absolute Error (MAE) - Lower is Better', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('MAE', fontsize=10)
axes[0, 1].tick_params(axis='x', rotation=45)
for i, v in enumerate(model_metrics_df['MAE']):
    axes[0, 1].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: R² Score
axes[1, 0].bar(model_metrics_df['Model'], model_metrics_df['R²'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 0].set_title('R² Score - Higher is Better', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('R² Score', fontsize=10)
axes[1, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(model_metrics_df['R²']):
    axes[1, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold', fontsize=9)
axes[1, 0].set_ylim([0, 1.0])
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Ranking Table
axes[1, 1].axis('off')
ranking_text = 'MODEL RANKING (by R² Score)\n' + '='*50 + '\n\n'
for idx, (i, row) in enumerate(model_metrics_df.iterrows()):
    ranking_text += f"{idx+1}. {row['Model']:20s} R²={row['R²']:.4f}  MAE={row['MAE']:.2f}  RMSE={row['RMSE']:.2f}\n"
ranking_text += '\n' + '='*50
axes[1, 1].text(0.05, 0.5, ranking_text, fontsize=9, verticalalignment='center', family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('model_performance_comparison.png', dpi=100, bbox_inches='tight')
plt.close()
print('\n✓ Performance comparison chart saved to: model_performance_comparison.png')


MODEL PERFORMANCE METRICS COMPARISON

OVERALL MODEL PERFORMANCE METRICS:
            Model     RMSE      MAE       R²
              MLP 4.920814 3.892639 0.944422
          XGBoost 4.997446 3.985009 0.942678
              GBM 5.001440 3.966183 0.942586
Linear Regression 5.192282 4.082219 0.938121
    Random Forest 5.628876 4.507703 0.927277

✓ Performance comparison chart saved to: model_performance_comparison.png


## Section 5: Model Prediction Trends Over Time

**Analyze:** How well each model tracks actual dengue cases over time

In [14]:
print('\n' + '='*100)
print('TIME-PERIOD TREND ANALYSIS')
print('='*100)

# Aggregate by Year-Month
test_data_full['YearMonth'] = test_data_full['Date'].dt.to_period('M')

time_trends = []
for idx, (ym, group) in enumerate(test_data_full.groupby('YearMonth')):
    actual_mean = group['Actual_Cases'].mean()
    month_data = {'YearMonth': str(ym), 'Month_Index': idx, 'Actual': actual_mean}
    for model in successfully_loaded:
        pred_col = f'{model}_Pred'
        month_data[model] = group[pred_col].mean()
    time_trends.append(month_data)

time_trends_df = pd.DataFrame(time_trends)

print(f'\nTime periods analyzed: {len(time_trends_df)}')
print(f'Date range: {time_trends_df["YearMonth"].iloc[0]} to {time_trends_df["YearMonth"].iloc[-1]}')

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Prediction trends over time
for model in successfully_loaded:
    axes[0].plot(time_trends_df['Month_Index'], time_trends_df[model], 
                marker='o', linewidth=2, label=model, alpha=0.8, markersize=4)
axes[0].plot(time_trends_df['Month_Index'], time_trends_df['Actual'], 
            marker='*', linewidth=3, color='black', label='Actual Cases', markersize=10)
axes[0].set_xlabel('Time Period', fontsize=11)
axes[0].set_ylabel('Average Dengue Cases', fontsize=11)
axes[0].set_title('Model Predictions vs Actual Cases Over Time', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9, loc='best')
axes[0].grid(True, alpha=0.3)

# Plot 2: Absolute errors over time
for model in successfully_loaded:
    errors = np.abs(time_trends_df['Actual'] - time_trends_df[model])
    axes[1].plot(time_trends_df['Month_Index'], errors, 
                marker='s', linewidth=2, label=model, alpha=0.8, markersize=4)
axes[1].set_xlabel('Time Period', fontsize=11)
axes[1].set_ylabel('Absolute Error (|Actual - Predicted|)', fontsize=11)
axes[1].set_title('Prediction Error Evolution Over Time', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('time_period_trends.png', dpi=100, bbox_inches='tight')
plt.close()
print('\n✓ Time-period trend chart saved to: time_period_trends.png')


TIME-PERIOD TREND ANALYSIS

Time periods analyzed: 34
Date range: 2021-03 to 2023-12

✓ Time-period trend chart saved to: time_period_trends.png


## Section 6: Ward-Level Model Consistency and Agreement Analysis

**Identify:** Which wards have consistent model predictions and which have high disagreement

In [15]:
print('\n' + '='*100)
print('WARD-LEVEL MODEL AGREEMENT ANALYSIS')
print('='*100)

# Calculate error metrics by ward for each model
ward_errors = []
for ward_id in test_data_full['Ward_ID'].unique():
    ward_data = test_data_full[test_data_full['Ward_ID'] == ward_id]
    actual = ward_data['Actual_Cases'].values
    ward_row = {'Ward_ID': int(ward_id)}
    
    for model in successfully_loaded:
        pred_col = f'{model}_Pred'
        predictions = ward_data[pred_col].values
        mae = mean_absolute_error(actual, predictions)
        r2 = r2_score(actual, predictions) if len(set(actual)) > 1 else 0
        ward_row[f'{model}_MAE'] = mae
        ward_row[f'{model}_R2'] = r2
    
    ward_errors.append(ward_row)

ward_errors_df = pd.DataFrame(ward_errors).sort_values('Ward_ID')

# Calculate agreement score (lower variance = better agreement)
ward_errors_df['MAE_Variance'] = ward_errors_df[[f'{m}_MAE' for m in successfully_loaded]].std(axis=1)
ward_errors_df['Agreement_Score'] = 1 / (1 + ward_errors_df['MAE_Variance'])

print(f'\n✓ Ward-level analysis for {len(ward_errors_df)} wards')
print(f'Average MAE variance across models per ward: {ward_errors_df["MAE_Variance"].mean():.3f}')
print(f'Average agreement score: {ward_errors_df["Agreement_Score"].mean():.3f}')

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Heatmap of MAE by ward and model
mae_cols = [f'{model}_MAE' for model in successfully_loaded]
ward_mae_pivot = ward_errors_df.set_index('Ward_ID')[mae_cols]
ward_mae_pivot.columns = successfully_loaded

im = axes[0, 0].imshow(ward_mae_pivot.T, aspect='auto', cmap='RdYlGn_r')
axes[0, 0].set_xlabel('Ward ID', fontsize=10)
axes[0, 0].set_ylabel('Model', fontsize=10)
axes[0, 0].set_title('MAE Heatmap - Yellow=Good, Red=Poor', fontsize=11, fontweight='bold')
axes[0, 0].set_yticks(range(len(successfully_loaded)))
axes[0, 0].set_yticklabels(successfully_loaded, fontsize=9)
axes[0, 0].set_xticks(range(0, len(ward_mae_pivot), max(1, len(ward_mae_pivot)//15)))
plt.colorbar(im, ax=axes[0, 0], label='MAE')

# Plot 2: Average MAE per model across wards
avg_mae_by_model = ward_errors_df[mae_cols].mean()
avg_mae_by_model.index = successfully_loaded
colors = plt.cm.Set3(np.linspace(0, 1, len(successfully_loaded)))
axes[0, 1].bar(range(len(successfully_loaded)), avg_mae_by_model.values, 
              color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 1].set_xticks(range(len(successfully_loaded)))
axes[0, 1].set_xticklabels(successfully_loaded, rotation=45, fontsize=9)
axes[0, 1].set_ylabel('Mean Absolute Error', fontsize=10)
axes[0, 1].set_title('Average MAE per Model Across All Wards', fontsize=11, fontweight='bold')
for i, v in enumerate(avg_mae_by_model.values):
    axes[0, 1].text(i, v + 0.1, f'{v:.1f}', ha='center', fontweight='bold', fontsize=8)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Model consistency (variance of MAE across wards)
mae_std_by_model = ward_errors_df[mae_cols].std()
mae_std_by_model.index = successfully_loaded
axes[1, 0].bar(range(len(successfully_loaded)), mae_std_by_model.values,
              color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 0].set_xticks(range(len(successfully_loaded)))
axes[1, 0].set_xticklabels(successfully_loaded, rotation=45, fontsize=9)
axes[1, 0].set_ylabel('Std Dev of MAE', fontsize=10)
axes[1, 0].set_title('Model Consistency Across Wards (Lower = More Consistent)', fontsize=11, fontweight='bold')
for i, v in enumerate(mae_std_by_model.values):
    axes[1, 0].text(i, v + 0.1, f'{v:.1f}', ha='center', fontweight='bold', fontsize=8)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Distribution of agreement scores
axes[1, 1].hist(ward_errors_df['Agreement_Score'], bins=15, alpha=0.7, color='steelblue', edgecolor='black')
axes[1, 1].axvline(ward_errors_df['Agreement_Score'].mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {ward_errors_df["Agreement_Score"].mean():.3f}')
axes[1, 1].axvline(ward_errors_df['Agreement_Score'].median(), color='green', linestyle='--',
                   linewidth=2, label=f'Median: {ward_errors_df["Agreement_Score"].median():.3f}')
axes[1, 1].set_xlabel('Agreement Score', fontsize=10)
axes[1, 1].set_ylabel('Frequency', fontsize=10)
axes[1, 1].set_title('Distribution of Ward-Level Model Agreement', fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ward_level_analysis.png', dpi=100, bbox_inches='tight')
plt.close()
print('\n✓ Ward-level analysis chart saved to: ward_level_analysis.png')


WARD-LEVEL MODEL AGREEMENT ANALYSIS

✓ Ward-level analysis for 40 wards
Average MAE variance across models per ward: 0.330
Average agreement score: 0.759

✓ Ward-level analysis chart saved to: ward_level_analysis.png


## Section 7: Time-Period Model Agreement Analysis

**Identify:** Which time periods (months) have consistent model predictions across all models

In [16]:
print('\n' + '='*100)
print('TIME-PERIOD MODEL AGREEMENT ANALYSIS')
print('='*100)

# Calculate error metrics by time period for each model
time_errors = []
for ym, group in test_data_full.groupby('YearMonth'):
    actual = group['Actual_Cases'].values
    time_row = {'YearMonth': str(ym)}
    
    for model in successfully_loaded:
        pred_col = f'{model}_Pred'
        predictions = group[pred_col].values
        mae = mean_absolute_error(actual, predictions)
        r2 = r2_score(actual, predictions) if len(set(actual)) > 1 else 0
        time_row[f'{model}_MAE'] = mae
        time_row[f'{model}_R2'] = r2
    
    time_errors.append(time_row)

time_errors_df = pd.DataFrame(time_errors)

# Calculate agreement score
time_errors_df['MAE_Variance'] = time_errors_df[[f'{m}_MAE' for m in successfully_loaded]].std(axis=1)
time_errors_df['Agreement_Score'] = 1 / (1 + time_errors_df['MAE_Variance'])

print(f'\n✓ Time-period analysis for {len(time_errors_df)} months')
print(f'Average MAE variance across models per month: {time_errors_df["MAE_Variance"].mean():.3f}')
print(f'Average agreement score: {time_errors_df["Agreement_Score"].mean():.3f}')

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Error trends for each model over time
mae_cols = [f'{model}_MAE' for model in successfully_loaded]
for model in successfully_loaded:
    axes[0, 0].plot(range(len(time_errors_df)), time_errors_df[f'{model}_MAE'],
                   marker='o', linewidth=2, label=model, alpha=0.8, markersize=4)
axes[0, 0].set_xlabel('Time Period Index', fontsize=10)
axes[0, 0].set_ylabel('Mean Absolute Error', fontsize=10)
axes[0, 0].set_title('Prediction Error Trends Over Time - All Models', fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=8, loc='best')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Average MAE per model
avg_mae_time = time_errors_df[mae_cols].mean()
avg_mae_time.index = successfully_loaded
colors = plt.cm.Set3(np.linspace(0, 1, len(successfully_loaded)))
axes[0, 1].bar(range(len(successfully_loaded)), avg_mae_time.values,
              color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 1].set_xticks(range(len(successfully_loaded)))
axes[0, 1].set_xticklabels(successfully_loaded, rotation=45, fontsize=9)
axes[0, 1].set_ylabel('Mean Absolute Error', fontsize=10)
axes[0, 1].set_title('Average MAE per Model Across All Time Periods', fontsize=11, fontweight='bold')
for i, v in enumerate(avg_mae_time.values):
    axes[0, 1].text(i, v + 0.1, f'{v:.1f}', ha='center', fontweight='bold', fontsize=8)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Variance of MAE (consistency over time)
std_mae_time = time_errors_df[mae_cols].std()
std_mae_time.index = successfully_loaded
axes[1, 0].bar(range(len(successfully_loaded)), std_mae_time.values,
              color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 0].set_xticks(range(len(successfully_loaded)))
axes[1, 0].set_xticklabels(successfully_loaded, rotation=45, fontsize=9)
axes[1, 0].set_ylabel('Std Dev of MAE', fontsize=10)
axes[1, 0].set_title('Temporal Consistency (Lower = More Stable Over Time)', fontsize=11, fontweight='bold')
for i, v in enumerate(std_mae_time.values):
    axes[1, 0].text(i, v + 0.1, f'{v:.1f}', ha='center', fontweight='bold', fontsize=8)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Time-period agreement scores over time
axes[1, 1].plot(range(len(time_errors_df)), time_errors_df['Agreement_Score'],
               marker='o', linewidth=2, color='steelblue', markersize=5, alpha=0.8)
axes[1, 1].axhline(time_errors_df['Agreement_Score'].mean(), color='red', linestyle='--',
                  linewidth=2, label=f'Mean: {time_errors_df["Agreement_Score"].mean():.3f}')
axes[1, 1].fill_between(range(len(time_errors_df)), time_errors_df['Agreement_Score'], 
                       time_errors_df['Agreement_Score'].mean(), alpha=0.2, color='steelblue')
axes[1, 1].set_xlabel('Time Period Index', fontsize=10)
axes[1, 1].set_ylabel('Agreement Score', fontsize=10)
axes[1, 1].set_title('Model Agreement Over Time (Higher = Better)', fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('time_period_agreement.png', dpi=100, bbox_inches='tight')
plt.close()
print('\n✓ Time-period agreement chart saved to: time_period_agreement.png')


TIME-PERIOD MODEL AGREEMENT ANALYSIS

✓ Time-period analysis for 34 months
Average MAE variance across models per month: 0.334
Average agreement score: 0.756

✓ Time-period agreement chart saved to: time_period_agreement.png


## Section 8: Best and Worst Performing Wards and Time Periods

**Summary:** Detailed identification of spatial and temporal patterns in model agreement

In [17]:
print('\n' + '='*100)
print('BEST AND WORST PERFORMING WARDS AND TIME PERIODS')
print('='*100)

# Create visualization for best/worst wards and time periods
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Top 15 wards with BEST model agreement
top_wards = ward_errors_df.nlargest(15, 'Agreement_Score')[['Ward_ID', 'Agreement_Score', 'MAE_Variance']]
axes[0, 0].barh(range(len(top_wards)), top_wards['Agreement_Score'].values, color='green', alpha=0.7, edgecolor='black')
axes[0, 0].set_yticks(range(len(top_wards)))
axes[0, 0].set_yticklabels([f"Ward {int(w)}" for w in top_wards['Ward_ID'].values], fontsize=9)
axes[0, 0].set_xlabel('Agreement Score', fontsize=10)
axes[0, 0].set_title('Top 15 Wards with BEST Model Agreement', fontsize=11, fontweight='bold')
for i, v in enumerate(top_wards['Agreement_Score'].values):
    axes[0, 0].text(v - 0.01, i, f'{v:.3f}', ha='right', va='center', fontweight='bold', fontsize=8)
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Plot 2: Top 15 wards with WORST model agreement
bottom_wards = ward_errors_df.nsmallest(15, 'Agreement_Score')[['Ward_ID', 'Agreement_Score', 'MAE_Variance']]
axes[0, 1].barh(range(len(bottom_wards)), bottom_wards['Agreement_Score'].values, color='red', alpha=0.7, edgecolor='black')
axes[0, 1].set_yticks(range(len(bottom_wards)))
axes[0, 1].set_yticklabels([f"Ward {int(w)}" for w in bottom_wards['Ward_ID'].values], fontsize=9)
axes[0, 1].set_xlabel('Agreement Score', fontsize=10)
axes[0, 1].set_title('Top 15 Wards with WORST Model Agreement', fontsize=11, fontweight='bold')
for i, v in enumerate(bottom_wards['Agreement_Score'].values):
    axes[0, 1].text(v - 0.01, i, f'{v:.3f}', ha='right', va='center', fontweight='bold', fontsize=8)
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Plot 3: Top 10 time periods with BEST model agreement
top_periods = time_errors_df.nlargest(10, 'Agreement_Score')[['YearMonth', 'Agreement_Score']]
axes[1, 0].barh(range(len(top_periods)), top_periods['Agreement_Score'].values, color='green', alpha=0.7, edgecolor='black')
axes[1, 0].set_yticks(range(len(top_periods)))
axes[1, 0].set_yticklabels(top_periods['YearMonth'].values, fontsize=8)
axes[1, 0].set_xlabel('Agreement Score', fontsize=10)
axes[1, 0].set_title('Top 10 Time Periods with BEST Model Agreement', fontsize=11, fontweight='bold')
for i, v in enumerate(top_periods['Agreement_Score'].values):
    axes[1, 0].text(v - 0.01, i, f'{v:.3f}', ha='right', va='center', fontweight='bold', fontsize=8)
axes[1, 0].grid(True, alpha=0.3, axis='x')

# Plot 4: Top 10 time periods with WORST model agreement
bottom_periods = time_errors_df.nsmallest(10, 'Agreement_Score')[['YearMonth', 'Agreement_Score']]
axes[1, 1].barh(range(len(bottom_periods)), bottom_periods['Agreement_Score'].values, color='red', alpha=0.7, edgecolor='black')
axes[1, 1].set_yticks(range(len(bottom_periods)))
axes[1, 1].set_yticklabels(bottom_periods['YearMonth'].values, fontsize=8)
axes[1, 1].set_xlabel('Agreement Score', fontsize=10)
axes[1, 1].set_title('Top 10 Time Periods with WORST Model Agreement', fontsize=11, fontweight='bold')
for i, v in enumerate(bottom_periods['Agreement_Score'].values):
    axes[1, 1].text(v - 0.01, i, f'{v:.3f}', ha='right', va='center', fontweight='bold', fontsize=8)
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('best_worst_analysis.png', dpi=100, bbox_inches='tight')
plt.close()
print('\n✓ Best/Worst analysis chart saved to: best_worst_analysis.png')

# Print detailed summary
print('\n' + '='*100)
print('WARD-LEVEL SUMMARY')
print('='*100)
print(f'\nBest Performing Wards (with agreement scores):')
print(ward_errors_df.nlargest(10, 'Agreement_Score')[['Ward_ID', 'Agreement_Score', 'MAE_Variance']].to_string(index=False))

print(f'\n\nWorst Performing Wards (with agreement scores):')
print(ward_errors_df.nsmallest(10, 'Agreement_Score')[['Ward_ID', 'Agreement_Score', 'MAE_Variance']].to_string(index=False))

print('\n' + '='*100)
print('TIME-PERIOD SUMMARY')
print('='*100)
print(f'\nBest Performing Time Periods (with agreement scores):')
print(time_errors_df.nlargest(10, 'Agreement_Score')[['YearMonth', 'Agreement_Score', 'MAE_Variance']].to_string(index=False))

print(f'\n\nWorst Performing Time Periods (with agreement scores):')
print(time_errors_df.nsmallest(10, 'Agreement_Score')[['YearMonth', 'Agreement_Score', 'MAE_Variance']].to_string(index=False))

# Save results
ward_errors_df.to_csv('ward_level_model_agreement.csv', index=False)
time_errors_df.to_csv('time_period_model_agreement.csv', index=False)

print('\n' + '='*100)
print('✓ Analysis complete! Files saved:')
print('  - model_comparison_ward_month_table.csv')
print('  - ward_level_model_agreement.csv')
print('  - time_period_model_agreement.csv')
print('  - model_performance_comparison.png')
print('  - time_period_trends.png')
print('  - ward_level_analysis.png')
print('  - time_period_agreement.png')
print('  - best_worst_analysis.png')
print('='*100)


BEST AND WORST PERFORMING WARDS AND TIME PERIODS

✓ Best/Worst analysis chart saved to: best_worst_analysis.png

WARD-LEVEL SUMMARY

Best Performing Wards (with agreement scores):
 Ward_ID  Agreement_Score  MAE_Variance
     176         0.904551      0.105521
     196         0.899240      0.112051
     195         0.884568      0.130495
     168         0.862566      0.159332
     190         0.856246      0.167889
     174         0.855374      0.169080
     169         0.827205      0.208891
     178         0.818882      0.221177
     166         0.816125      0.225303
     162         0.807908      0.237765


Worst Performing Wards (with agreement scores):
 Ward_ID  Agreement_Score  MAE_Variance
     172         0.598572      0.670643
     185         0.635802      0.572816
     171         0.638348      0.566543
     164         0.646229      0.547440
     160         0.673535      0.484703
     181         0.677156      0.476765
     198         0.678990      0.472776
     194 

## Summary of Key Findings

### What This Analysis Shows:

1. **Model Performance Comparison**: Overall metrics (RMSE, MAE, R²) across all models to identify the best-performing model

2. **Temporal Trends**: How prediction accuracy changes over time - identifying months where all models agree vs. disagree

3. **Ward-Level Analysis**: Which wards have consistent predictions across models (high agreement) and which have divergent predictions (low agreement)

4. **Spatial Patterns**: Geographic variation in model performance - showing which wards are easier/harder to predict

5. **Best/Worst Conditions**:
   - **Best wards**: Where models make consistent predictions (small variance in errors)
   - **Worst wards**: Where models diverge significantly (large variance in errors)
   - **Best periods**: Months where prediction uncertainty is lowest
   - **Worst periods**: Months with high model disagreement

### Actionable Insights:
- Use ensemble predictions for wards with poor agreement to reduce uncertainty
- Focus model improvements on worst-performing wards and time periods
- Leverage high-agreement wards for gaining confidence in predictions
- Consider temporal patterns when making seasonal predictions
